# Image contrast - single-image contrast metrics

## Metrics implemented (and where they come from)
- **RMS contrast** (default, assumption-free): the standard deviation of intensities, and
  the coefficient of variation (std / mean). Robust global measure that needs no structure
  assumptions. *Peli 1990, "Contrast in complex images", JOSA A.*
- **Michelson contrast** (visibility): `(I_high - I_low) / (I_high + I_low)`, evaluated on
  robust intensity percentiles so single hot/dead pixels do not dominate. Classic measure
  for the visibility of light vs dark regions. *Michelson 1927.*
- **EM staining contrast** (the EM-specific measure): `(I_high - I_low) / (mean(I_high,I_low) - I0)`
  where `I_high`/`I_low` are the strongly- and weakly-stained intensity levels and `I0` is the
  detector dark count. `I_high`/`I_low` are read either from the two peaks of the intensity
  histogram (bimodal case) or from the 75%/25% CDF levels (robust fallback). This is the
  contrast definition proposed for FIB-SEM staining assessment. *Shtengel et al. 2026,
  "Image Analysis Tools for Electron Microscopy" (in `reference_materials/MArch_2026/`).*

Full citations are in the **References** cell at the bottom.

## Configuration
- **`INPUT_PATH`**: folder containing input `.tif` images (one image per field of view).
- **`OUTPUT_PATH`**: folder for the CSV output (leave `""` to skip writing).
- **`CELL_MASK_DIR` / `CELL_MASK_TOKEN`**: per-image mask `<raw_stem><token>.tif`. Leave
  `CELL_MASK_DIR = ""` to measure whole-image contrast only.
- **`DARK_COUNT_I0`**: detector dark count (zero-signal offset) used by the EM staining
  contrast. `0.0` if unknown (then staining contrast reduces to a range/mean ratio). If you
  have run the SNR/noise analysis, use the `I0` it reports.
- **`GRADIENT_KEEP_FRACTION`**: for the EM staining contrast, keep only this fraction of
  pixels with the *lowest* local gradient before finding `I_high`/`I_low`. This excludes
  edge/transition pixels so the two staining levels separate cleanly (Shtengel et al. use
  0.25). Set to `1.0` to disable.
- **`STAIN_METHOD`**: `"auto"` (try bimodal histogram peaks, fall back to percentiles) |
  `"percentile"` (always use CDF levels) | `"bimodal"` (double-Gaussian fit; needs SciPy).
- **`CDF_LOW` / `CDF_HIGH`**: CDF levels (%) for the percentile `I_low`/`I_high` (default 25/75).
- **`MICHELSON_CLIP`**: percentile clip (%) for robust Michelson `I_low`/`I_high` (default 1/99).

In [ ]:
INPUT_PATH  = r"../data/images"   # folder with input .tif images (single image per FOV)
OUTPUT_PATH = r"../outputs/contrast"   # folder for CSV output (leave "" to skip writing)

# Cell-mask support (same convention as the sharpness / FRC notebooks).
# For each raw <stem>.tif, the mask is <stem><CELL_MASK_TOKEN>.tif found under CELL_MASK_DIR.
CELL_MASK_DIR   = r"../data/masks"   # leave "" to measure whole-image contrast only
CELL_MASK_TOKEN = "_mitochondria"

# EM staining-contrast parameters
DARK_COUNT_I0          = 0.0       # detector dark count / zero-signal offset (0.0 if unknown)
GRADIENT_KEEP_FRACTION = 0.25      # keep lowest-gradient fraction before finding I_high/I_low; 1.0 = off
STAIN_METHOD           = "percentile"    # "auto" | "percentile" | "bimodal"
CDF_LOW, CDF_HIGH      = 25.0, 75.0   # CDF levels (%) for percentile I_low / I_high
MICHELSON_CLIP         = (1.0, 99.0)  # robust percentile clip (%) for Michelson

MIN_REGION_PX = 64     # skip images / masked regions smaller than this (pixels)

print("Input folder     :", INPUT_PATH)
print("Output folder    :", OUTPUT_PATH)
print("Cell-mask folder :", CELL_MASK_DIR or "(none - whole image only)")
print("Dark count I0    :", DARK_COUNT_I0)
print("Gradient keep    :", GRADIENT_KEEP_FRACTION)
print("Staining method  :", STAIN_METHOD)

In [ ]:
import os
import csv

import numpy as np
import tifffile
import matplotlib.pyplot as plt

# SciPy is optional: only used for the double-Gaussian ("bimodal") staining fit.
try:
    from scipy.optimize import curve_fit
    _HAVE_SCIPY = True
except Exception:
    _HAVE_SCIPY = False

## Pixel selection helpers
Contrast is computed on a set of intensity values. `gather_values` returns the intensities
inside an optional boolean mask. `low_gradient_mask` flags the fraction of pixels with the
lowest local gradient magnitude - used by the EM staining contrast to drop edge/transition
pixels so the strongly- and weakly-stained intensity levels separate cleanly.

In [ ]:
def gather_values(image, mask=None):
    """1-D array of intensities inside `mask` (or the whole image if mask is None)."""
    img = np.asarray(image, dtype=np.float64)
    if mask is None:
        return img.ravel()
    m = np.asarray(mask) > 0
    return img[m]


def low_gradient_mask(image, keep_fraction=1.0, base_mask=None):
    """Boolean mask of the `keep_fraction` of pixels with the lowest gradient magnitude.

    Excludes sharp edges/transitions so intensity-level estimates are not blurred across
    boundaries. `base_mask` (if given) restricts the selection to those pixels. With
    keep_fraction >= 1.0 the (base) mask is returned unchanged.
    """
    img = np.asarray(image, dtype=np.float64)
    valid = np.ones(img.shape, dtype=bool) if base_mask is None else (np.asarray(base_mask) > 0)
    if keep_fraction >= 1.0:
        return valid

    gy, gx = np.gradient(img)
    grad = np.sqrt(gx ** 2 + gy ** 2)
    vals = grad[valid]
    if vals.size == 0:
        return valid
    thresh = np.quantile(vals, keep_fraction)
    return valid & (grad <= thresh)

## RMS contrast
The root-mean-square contrast is simply the **standard deviation of the intensities** - the
most widely used, assumption-free global contrast measure (Peli 1990). It is reported both
as the raw standard deviation (`rms_std`, in intensity units) and as the **coefficient of
variation** `std / mean` (`rms_contrast`), which is dimensionless and therefore comparable
across images with different brightness / bit depth.

In [ ]:
def rms_contrast(values):
    """RMS contrast. Returns (rms_std, rms_cv) where rms_cv = std / mean (coeff. of variation)."""
    v = np.asarray(values, dtype=np.float64)
    if v.size == 0:
        return None, None
    std = float(np.std(v))
    mean = float(np.mean(v))
    cv = std / mean if abs(mean) > 1e-12 else None
    return std, cv

## Michelson contrast (visibility)
`(I_high - I_low) / (I_high + I_low)`, the classic visibility measure, bounded in `[0, 1]`
for non-negative data. Raw min/max are extremely sensitive to a single hot or dead pixel, so
`I_low`/`I_high` are taken as robust percentiles (default 1st / 99th, set by `MICHELSON_CLIP`).

In [ ]:
def michelson_contrast(values, clip=(1.0, 99.0)):
    """Robust Michelson contrast (I_high - I_low) / (I_high + I_low) on clipped percentiles."""
    v = np.asarray(values, dtype=np.float64)
    if v.size == 0:
        return None
    lo = np.percentile(v, clip[0])
    hi = np.percentile(v, clip[1])
    denom = hi + lo
    if abs(denom) < 1e-12:
        return None
    return float((hi - lo) / denom)

## EM staining contrast (Shtengel et al. 2026)
The EM-specific measure:
$$\mathrm{Contrast} = \frac{I_{high} - I_{low}}{\tfrac{1}{2}(I_{high}+I_{low}) - I_0}$$
where `I_high` is the average signal in strongly-stained areas (e.g. membranes), `I_low` the
signal in weakly-stained areas (e.g. intercellular space), and `I0` the detector dark count.
Dividing by the mean level *minus the dark count* makes the measure a true modulation about
the real zero-signal level (not an arbitrary saved offset).

`I_high` / `I_low` are estimated in one of two ways:
- **percentile** (robust default): the `CDF_HIGH` / `CDF_LOW` levels of the intensity CDF
  (75% / 25%). This is the paper's fallback for when the histogram is not clearly bimodal.
- **bimodal**: fit a **double Gaussian** to the intensity histogram and take the two peak
  centres (needs SciPy). Best when the sample really has two stain populations.

`"auto"` tries the bimodal fit and falls back to percentiles if it fails or is not clearly
bimodal. High-gradient (edge) pixels are optionally dropped first (`GRADIENT_KEEP_FRACTION`)
so the two levels are not smeared across boundaries.

In [ ]:
def _double_gaussian(x, a1, m1, s1, a2, m2, s2):
    return (a1 * np.exp(-0.5 * ((x - m1) / s1) ** 2)
            + a2 * np.exp(-0.5 * ((x - m2) / s2) ** 2))


def bimodal_levels(values, bins=256):
    """I_low, I_high from a double-Gaussian fit of the intensity histogram.

    Returns a dict (i_low, i_high, r2, separated, balanced) or None if SciPy is missing or
    the fit cannot be computed. `separated`/`balanced` flag whether the two Gaussians are a
    genuine two-population split (well apart, and both carrying real weight) rather than two
    components collapsing onto the same peak.
    """
    if not _HAVE_SCIPY:
        return None
    v = np.asarray(values, dtype=np.float64)
    if v.size < 50:
        return None

    counts, edges = np.histogram(v, bins=bins, density=True)
    centres = 0.5 * (edges[:-1] + edges[1:])

    # Push the two component means apart initially so the fit does not collapse to one peak.
    p10, p25, p75, p90 = np.percentile(v, [10, 25, 75, 90])
    spread = max((p75 - p25) / 2.0, 1e-6)
    amp = counts.max() if counts.max() > 0 else 1.0
    p0 = [amp, p10, spread, amp, p90, spread]
    lo_b = [0.0, centres[0], 1e-6, 0.0, centres[0], 1e-6]
    hi_b = [np.inf, centres[-1], centres[-1] - centres[0],
            np.inf, centres[-1], centres[-1] - centres[0]]
    try:
        popt, _ = curve_fit(_double_gaussian, centres, counts, p0=p0,
                            bounds=(lo_b, hi_b), maxfev=10000)
    except Exception:
        return None

    a1, m1, s1, a2, m2, s2 = popt
    (i_low, s_low, a_low), (i_high, s_high, a_high) = sorted(
        ((float(m1), abs(s1), abs(a1)), (float(m2), abs(s2), abs(a2))), key=lambda t: t[0])
    if i_high - i_low < 1e-9:
        return None

    fit = _double_gaussian(centres, *popt)
    ss_res = float(np.sum((counts - fit) ** 2))
    ss_tot = float(np.sum((counts - counts.mean()) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0

    # Genuine bimodality: peaks separated by more than their combined half-width, and each
    # component carrying non-trivial weight (guards against both Gaussians on one peak).
    separated = (i_high - i_low) > 0.5 * (s_low + s_high)
    peak_low = a_low / (s_low + 1e-12)
    peak_high = a_high / (s_high + 1e-12)
    balanced = min(peak_low, peak_high) / max(peak_low, peak_high, 1e-12) > 0.1

    return {"i_low": i_low, "i_high": i_high, "r2": r2,
            "separated": bool(separated), "balanced": bool(balanced)}


def staining_contrast(image, base_mask=None, i0=0.0, keep_fraction=1.0,
                      cdf_low=25.0, cdf_high=75.0, method="auto"):
    """EM staining contrast (Shtengel et al. 2026).

    Returns a dict with i_low, i_high, i0, contrast, and the level-estimation method used.
    """
    grad_mask = low_gradient_mask(image, keep_fraction=keep_fraction, base_mask=base_mask)
    values = gather_values(image, grad_mask)
    if values.size == 0:
        return None

    used = "percentile"
    levels = None
    if method in ("auto", "bimodal"):
        fit = bimodal_levels(values)
        # "bimodal": take the fit whenever it is computable.
        # "auto": only trust a genuine, well-resolved two-population fit; otherwise fall
        #         back to percentiles (many EM crops are not cleanly bimodal).
        if fit is not None and (
            method == "bimodal"
            or (fit["r2"] >= 0.8 and fit["separated"] and fit["balanced"])
        ):
            levels = (fit["i_low"], fit["i_high"])
            used = "bimodal"

    if levels is None:
        i_low = float(np.percentile(values, cdf_low))
        i_high = float(np.percentile(values, cdf_high))
        levels = (i_low, i_high)
        used = "percentile"

    i_low, i_high = levels
    denom = 0.5 * (i_high + i_low) - i0
    contrast = float((i_high - i_low) / denom) if abs(denom) > 1e-12 else None
    return {"i_low": i_low, "i_high": i_high, "i0": float(i0),
            "contrast": contrast, "method": used}

## High-level single-image API
`measure_contrast` computes all three metrics on one image (optionally restricted to a cell
mask) and returns them in a single dict, together with the intensity values needed for plotting.

In [ ]:
def measure_contrast(image, mask=None, i0=DARK_COUNT_I0,
                     keep_fraction=GRADIENT_KEEP_FRACTION, stain_method=STAIN_METHOD,
                     cdf_low=CDF_LOW, cdf_high=CDF_HIGH, michelson_clip=MICHELSON_CLIP):
    """Compute RMS, Michelson and EM staining contrast for one image (optionally masked)."""
    values = gather_values(image, mask)
    if values.size == 0:
        return None

    rms_std, rms_cv = rms_contrast(values)
    michelson = michelson_contrast(values, clip=michelson_clip)
    stain = staining_contrast(image, base_mask=mask, i0=i0, keep_fraction=keep_fraction,
                              cdf_low=cdf_low, cdf_high=cdf_high, method=stain_method)

    return {
        "n_pixels": int(values.size),
        "mean": float(np.mean(values)),
        "rms_std": rms_std,
        "rms_contrast": rms_cv,
        "michelson_contrast": michelson,
        "staining_contrast": stain["contrast"] if stain else None,
        "staining_i_low": stain["i_low"] if stain else None,
        "staining_i_high": stain["i_high"] if stain else None,
        "staining_method": stain["method"] if stain else None,
        "i0": float(i0),
        "_values": values,
    }


def measure_contrast_in_mask(image, mask, min_size=MIN_REGION_PX, **kwargs):
    """Contrast measured only inside a cell mask; None if the mask is too small."""
    m = np.asarray(mask) > 0
    if m.sum() < min_size:
        return None
    return measure_contrast(image, mask=m, **kwargs)

## Mask discovery and file loading
Same conventions as the sharpness / FRC notebooks: for a raw `<stem>.tif`, the mask is
`<stem><CELL_MASK_TOKEN>.tif` located anywhere under `CELL_MASK_DIR`. Obvious mask /
probability companion files are skipped when listing raw images.

In [ ]:
def find_cell_mask(raw_path, cell_mask_dir, token=CELL_MASK_TOKEN):
    """Locate <raw_stem><token>.tif under cell_mask_dir (recursive). Returns None if absent."""
    if not cell_mask_dir:
        return None
    stem = os.path.splitext(os.path.basename(raw_path))[0]
    target = stem + token + ".tif"
    for dirpath, _, filenames in os.walk(cell_mask_dir):
        if target in filenames:
            return os.path.join(dirpath, target)
    return None


def load_cell_mask(cell_mask_path):
    """Load a cell mask as a boolean array (any non-zero pixel = inside cell)."""
    cm = tifffile.imread(cell_mask_path)
    if cm.ndim > 2:
        cm = cm[0] if cm.shape[0] < cm.shape[-1] else cm[..., 0]
    return cm > 0


SKIP_TOKENS = ("_full_cell", "_Probabilities", "_Simple Segmentation", "_mask")


def find_tif_files(root):
    """All .tif/.tiff files under root, excluding obvious mask / probability companions."""
    paths = []
    for dirpath, _, filenames in os.walk(root):
        for fname in filenames:
            if not fname.lower().endswith((".tif", ".tiff")):
                continue
            if any(tok in fname for tok in SKIP_TOKENS):
                continue
            paths.append(os.path.join(dirpath, fname))
    return sorted(paths)


def load_image_2d(path):
    """Load a TIFF as a 2-D float array (middle slice of a stack; first channel of RGB)."""
    img = tifffile.imread(path)
    if img.ndim == 2:
        return img.astype(np.float64)
    if img.ndim == 3:
        if img.shape[-1] <= 4:
            return img[..., 0].astype(np.float64)
        return img[img.shape[0] // 2].astype(np.float64)
    raise ValueError(f"Unsupported image shape {img.shape} for {path}")

## Plotting
Show the image (masked region highlighted) alongside its intensity histogram (PDF) and CDF,
with the `I_low`, `I_high` and `I0` levels marked - the same view used in Shtengel et al.
Figure 2 to read off the staining contrast.

In [ ]:
def plot_contrast(image, result, mask=None, title="", bins=256):
    """Image + intensity histogram/CDF with I_low / I_high / I0 marked."""
    values = result["_values"]
    fig, (ax_img, ax_hist) = plt.subplots(1, 2, figsize=(12, 4.6))

    disp = np.asarray(image, dtype=np.float64)
    vlo, vhi = np.percentile(values, [1, 99])
    ax_img.imshow(disp, cmap="gray", vmin=vlo, vmax=vhi)
    if mask is not None:
        ax_img.contour(np.asarray(mask) > 0, levels=[0.5], colors="yellow", linewidths=0.8)
    ax_img.set_title(title or "image")
    ax_img.axis("off")

    counts, edges = np.histogram(values, bins=bins, density=True)
    centres = 0.5 * (edges[:-1] + edges[1:])
    ax_hist.plot(centres, counts, lw=1.5, color="steelblue", label="PDF")
    ax_hist.fill_between(centres, counts, color="steelblue", alpha=0.15)

    i_low = result.get("staining_i_low")
    i_high = result.get("staining_i_high")
    i0 = result.get("i0")
    if i_low is not None:
        ax_hist.axvline(i_low, color="navy", ls="--", lw=1.3, label=f"I_low = {i_low:.1f}")
    if i_high is not None:
        ax_hist.axvline(i_high, color="crimson", ls="--", lw=1.3, label=f"I_high = {i_high:.1f}")
    if i0 is not None and i0 != 0.0:
        ax_hist.axvline(i0, color="grey", ls=":", lw=1.2, label=f"I0 = {i0:.1f}")

    ax_cdf = ax_hist.twinx()
    order = np.argsort(values)
    sv = values[order]
    cdf = np.arange(1, sv.size + 1) / sv.size
    ax_cdf.plot(sv, cdf, color="darkorange", lw=1.2, alpha=0.8, label="CDF")
    ax_cdf.set_ylabel("CDF", color="darkorange")
    ax_cdf.set_ylim(0, 1)

    sc = result.get("staining_contrast")
    rc = result.get("rms_contrast")
    mc = result.get("michelson_contrast")
    txt = []
    if sc is not None:
        txt.append(f"staining={sc:.3f}")
    if mc is not None:
        txt.append(f"michelson={mc:.3f}")
    if rc is not None:
        txt.append(f"rms(cv)={rc:.3f}")
    ax_hist.set_title("  |  ".join(txt))
    ax_hist.set_xlabel("Intensity")
    ax_hist.set_ylabel("PDF", color="steelblue")
    ax_hist.legend(loc="upper right", fontsize=8)
    fig.tight_layout()
    return fig, (ax_img, ax_hist)

## Batch over a folder
Walk `INPUT_PATH`, compute whole-image contrast for every `.tif`, and (when a matching cell
mask exists) the inside-mask contrast too. Results are collected into a table and written to
`contrast_results.csv` when `OUTPUT_PATH` is set.

In [ ]:
records = []
tif_paths = find_tif_files(INPUT_PATH) if INPUT_PATH else []
print(f"Found {len(tif_paths)} image(s) under {INPUT_PATH}\n")

for path in tif_paths:
    row = {"File": os.path.basename(path), "Path": path,
           "RMS_Contrast": np.nan, "Michelson_Contrast": np.nan, "Staining_Contrast": np.nan,
           "Mask_RMS_Contrast": np.nan, "Mask_Michelson_Contrast": np.nan,
           "Mask_Staining_Contrast": np.nan, "MaskFound": False}
    try:
        image = load_image_2d(path)
        whole = measure_contrast(image)
        if whole is not None:
            row["RMS_Contrast"] = whole["rms_contrast"]
            row["Michelson_Contrast"] = whole["michelson_contrast"]
            row["Staining_Contrast"] = whole["staining_contrast"]

        mask_path = find_cell_mask(path, CELL_MASK_DIR)
        if mask_path:
            mask = load_cell_mask(mask_path)
            if mask.shape == image.shape:
                res_m = measure_contrast_in_mask(image, mask)
                if res_m is not None:
                    row["MaskFound"] = True
                    row["Mask_RMS_Contrast"] = res_m["rms_contrast"]
                    row["Mask_Michelson_Contrast"] = res_m["michelson_contrast"]
                    row["Mask_Staining_Contrast"] = res_m["staining_contrast"]
    except Exception as exc:
        print(f"  FAILED {path}: {exc}")

    records.append(row)
    print("--------------")
    print(path)
    print(f"  whole image : rms(cv)={row['RMS_Contrast']}  michelson={row['Michelson_Contrast']}  staining={row['Staining_Contrast']}")
    if row["MaskFound"]:
        print(f"  inside mask : rms(cv)={row['Mask_RMS_Contrast']}  michelson={row['Mask_Michelson_Contrast']}  staining={row['Mask_Staining_Contrast']}")
    print("--------------")

if OUTPUT_PATH and records:
    os.makedirs(OUTPUT_PATH, exist_ok=True)
    csv_path = os.path.join(OUTPUT_PATH, "contrast_results.csv")
    fields = ["File", "Path", "RMS_Contrast", "Michelson_Contrast", "Staining_Contrast",
              "Mask_RMS_Contrast", "Mask_Michelson_Contrast", "Mask_Staining_Contrast", "MaskFound"]
    with open(csv_path, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=fields)
        w.writeheader()
        w.writerows(records)
    print(f"\nCSV -> {csv_path}")